# RoseTTAFold3 Structure Prediction

![RoseTTAFold3 Structure Prediction](https://proto-bio.github.io/proto-assets/images/tool/rf3/hero.png)

This notebook demonstrates all-atom structure prediction with RoseTTAFold3 (RF3) from the [Institute for Protein Design](https://www.ipd.uw.edu/). RF3 predicts 3D structures of proteins, DNA, RNA, small-molecule ligands, and their complexes via a diffusion-based generative model with explicit chirality handling. We walk through `run_rf3_prediction` on a small protein-ligand complex, covering input construction, configuration, result inspection, and export.

In [1]:
from proto_tools.utils.notebook_docs import (
    display_api_reference,
    display_available_tools,
    display_doc_link,
    display_docs_section,
    display_overview,
)

display_doc_link("rf3")
display_overview("rf3")
display_docs_section("rf3", "Background")

# RoseTTAFold3 (RF3)

RoseTTAFold3 (RF3) is an all-atom biomolecular structure prediction model from the [Institute for Protein Design](https://www.ipd.uw.edu/) at the University of Washington, distributed as part of the [RosettaCommons Foundry](https://github.com/RosettaCommons/foundry) framework. It predicts the joint 3D structure of complexes that combine proteins, DNA, RNA, and small-molecule ligands, with first-class support for chirality. This toolkit runs RF3 structure prediction from sequences, SMILES, and CCD codes, with optional ColabFold multiple-sequence alignments.

RoseTTAFold3 ([Corley et al., 2025](https://doi.org/10.1101/2025.08.14.670328)) predicts the joint 3D structure of a biomolecular assembly from the sequences and chemical components it contains. It is the latest entry in the RoseTTAFold lineage from the Baker and DiMaio labs at the Institute for Protein Design, and like AlphaFold3 and Boltz-2 it folds proteins, nucleic acids, and small-molecule ligands within a single model. The preprint reports that an improved treatment of chirality narrows the performance gap between RF3 and the closed-source AlphaFold3 on biomolecular benchmarks.

Architecturally RF3 builds on the new [AtomWorks](https://github.com/RosettaCommons/atomworks) data framework introduced alongside it in the preprint, and uses an AlphaFold3 style trunk together with a diffusion module that samples several candidate structures per complex from random noise. The best sample is selected by a composite ranking score that combines interface pTM, overall pTM, and a clash penalty. Alongside the predicted coordinates, RF3 reports per-residue and overall pLDDT, per-chain confidence, predicted aligned error (PAE) and predicted distance error (PDE), chain-pair PAE and PDE matrices for multi-chain inputs, and a boolean flag for steric clashes.

The reference implementation is open-sourced as part of the [RosettaCommons/foundry](https://github.com/RosettaCommons/foundry) monorepo under the BSD-3-Clause license, with model weights served openly from the IPD file server. It was developed at the [Institute for Protein Design](https://www.ipd.uw.edu/) (UW).

## Available tools

In [2]:
display_available_tools("rf3")

- **`run_rf3_prediction()`** — All-atom structure prediction with explicit chirality (RoseTTAFold3)

### `run_rf3_prediction`

Predicts the joint 3D structure of a biomolecular complex. RF3's distinguishing capabilities are explicit chirality handling and optional ligand-conformer conditioning; outputs include per-chain pTM, chain-pair PAE / PDE matrices for multi-chain complexes, and a composite `ranking_score`.

In [3]:
from pathlib import Path

from proto_tools import Chain, Complex, RF3Config, RF3Input, run_rf3_prediction
from proto_tools.entities.ligands import Fragment

In [4]:
display_api_reference("rf3-prediction", "input", "run_rf3_prediction")

# Cro repressor from bacteriophage lambda (short, well-folded test protein)
cro_sequence = "MQTQNNSREKQAAALERLFLSCFLKDPVPKPLQEGTCDDVLCRELLNESETHLVQSIFRKESKVPGA"
# L-tyrosine SMILES (resolves to CCD "TYR")
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

complex = Complex(
    chains=[
        Chain(sequence=cro_sequence, entity_type="protein"),
        Fragment(smiles=tyrosine_smiles),
    ]
)

inputs = RF3Input(complexes=[complex])

**Input** — `RF3Input`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `complexes` | `list[proto_tools.entities.complex.Complex]` | required | List of complexes to predict structure for containing chains and entity types. |
| `msas` | `list[proto_tools.tools.structure_prediction.shared_data_models.ComplexMSAs] \| None` | `None` | Per-complex MSAs; a bare dict[int, MSA] is coerced to an unpaired ComplexMSAs. |

In [5]:
display_api_reference("rf3-prediction", "config", "run_rf3_prediction")

# Reduced settings for a fast demonstration run.
config = RF3Config(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available (unsupported in practice)
    use_msa=False,
    diffusion_batch_size=1,
)

**Config** — `RF3Config`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on (e.g., 'cuda', 'cpu') |
| `timeout` | `int \| None` | `1800` | Maximum execution time in seconds. RF3 is heavier than Boltz2; default is set accordingly. |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `include_pae_matrix` | `bool` | `False` | Attach the full per-residue PAE matrix. |
| `use_msa` | `bool` | `True` | Whether to auto-generate MSAs via MMseqs2 homology search; supplied MSAs are always used |
| `msa_search_config` | `proto_tools.tools.sequence_alignment.mmseqs2.homology_search.Mmseqs2HomologySearchConfig \| None` | `None` | Nested MMseqs2 homology-search config for MSA generation; None uses default settings. |
| `pair_heterocomplex_msas` | `bool` | `True` | Whether heterocomplex protein chains should use taxonomy-paired MSA generation. |
| `n_recycles` | `int` | `10` | Iterative refinement passes through the network. Higher = more accurate but slower. |
| `diffusion_batch_size` | `int` | `5` | Independent diffusion samples per complex; the best by ranking_score is kept. |
| `num_steps` | `int` | `50` | Denoising steps in the diffusion process. Higher = more refined but slower. |
| `cyclic_chains` | `list[str]` | `[]` | Chain IDs (e.g. ['A']) to mark as cyclic. |

In [6]:
result = run_rf3_prediction(inputs, config)

Folding structures (RF3):   0%|          | 0/1 [00:00<?, ?complex/s]

First-time setup for rf3. Installing dependencies. This is a one-time process; subsequent runs will start much faster.


Removed 1 proto-tools env registrations from /home/user/.conda/environments.txt


In [7]:
display_api_reference("rf3-prediction", "output", "run_rf3_prediction")

structure = result.structures[0]
m = structure.metrics

print(f"  Number of chains: {len(complex.chains)}")
print(f"  Protein length:   {len(cro_sequence)} residues")
print(f"  ranking_score:    {m.ranking_score:.3f}")
print(f"  avg_plddt:        {m.avg_plddt:.3f}")
print(f"  ptm:              {m.ptm:.3f}")
print(f"  avg_pae:          {m.avg_pae:.3f} Å")
print(f"  pde:              {m.pde:.3f} Å")
print(f"  chain_ptm:        {m.chain_ptm}")
print(f"  has_clash:        {m.has_clash}")

**Output** — `RF3Output`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `structures` | `list[proto_tools.entities.structures.structure.Structure]` | required | List of predicted structures, one per input complex. |

  Number of chains: 2
  Protein length:   67 residues
  ranking_score:    0.387
  avg_plddt:        0.730
  ptm:              0.498
  avg_pae:          10.588 Å
  pde:              3.117 Å
  chain_ptm:        [0.73, 0.58]
  has_clash:        False


#### Visualize the predicted complex

> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [8]:
structure.visualize(style="cartoon", color_by="bfactor")

## Export Results

Predicted structures can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD. The B-factor column carries pLDDT scores for per-residue confidence visualization.

In [9]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="rf3_cro_tyrosine", export_path=output_dir, file_format="pdb")
print(f"Structure exported to {output_dir / 'rf3_cro_tyrosine.pdb'}")

Structure exported to example_output/rf3_cro_tyrosine.pdb
